In [3]:
"""
Stage 2 - Stratified Sampling and Sentence Embedding
"""

import pandas as pd
import numpy as np
import torch
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import time

# Paths
PROCESSED   = Path("../data/processed/cicids_processed.csv")
OUTPUT_DIR  = Path("../embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_CSV  = OUTPUT_DIR / "stratified_sample.csv"
EMBEDDINGS  = OUTPUT_DIR / "embeddings.npy"
METADATA    = OUTPUT_DIR / "embedding_metadata.json"

# Parameters
SAMPLE_PER_TACTIC = 2000   
BENIGN_SAMPLE     = 2000   
RANDOM_SEED       = 42     
BATCH_SIZE        = 128    
MODEL_NAME        = "all-MiniLM-L6-v2"

# Load processed data
def load_data():
    print("Loading processed CIC-IDS data...")
    df = pd.read_csv(PROCESSED, low_memory=False, index_col="alert_id")
    print(f"  Total alerts available: {len(df):,}")
    print(f"  Tactics available:")
    print(df["attck_tactic"].value_counts().to_string())
    return df

# Stratified sampling
def stratified_sample(df):
    """
    Sample SAMPLE_PER_TACTIC alerts from each attack tactic.
    If a tactic has fewer than SAMPLE_PER_TACTIC alerts, take all of them.
    Benign alerts sampled separately.
    """
    print(f"\nBuilding stratified sample (seed={RANDOM_SEED})...")

    attack_frames = []
    attack_df = df[df["attck_tactic"] != "Benign"].copy()
    tactics   = attack_df["attck_tactic"].unique()

    print(f"\n  Attack tactics to sample:")
    for tactic in sorted(tactics):
        tactic_df = attack_df[attack_df["attck_tactic"] == tactic]
        available = len(tactic_df)
        n         = min(available, SAMPLE_PER_TACTIC)
        sampled   = tactic_df.sample(n=n, random_state=RANDOM_SEED)
        attack_frames.append(sampled)
        print(f"    {tactic:30s}: {available:6,} available -> {n:5,} sampled")

    # Benign sample
    benign_df = df[df["attck_tactic"] == "Benign"]
    n_benign  = min(len(benign_df), BENIGN_SAMPLE)
    benign_sampled = benign_df.sample(n=n_benign, random_state=RANDOM_SEED)
    print(f"    {'Benign':30s}: {len(benign_df):6,} available -> {n_benign:5,} sampled")

    # Combine and shuffle
    sample = pd.concat(attack_frames + [benign_sampled], ignore_index=True)
    sample = sample.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    sample.index.name = "sample_id"

    print(f"\n  Total sample size: {len(sample):,}")
    print(f"  Final tactic distribution:")
    print(sample["attck_tactic"].value_counts().to_string())

    # Save sample
    sample.to_csv(SAMPLE_CSV)
    print(f"\n  Sample saved to: {SAMPLE_CSV}")
    return sample

# Sentence embedding
def embed_alerts(sample):
    """
    Embed all alert_text strings using all-MiniLM-L6-v2.
    """
    # Detect device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"\nEmbedding on GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB VRAM)")
    else:
        print(f"\nEmbedding on CPU (no CUDA detected)")

    # Load model
    print(f"Loading model: {MODEL_NAME}")
    model = SentenceTransformer(MODEL_NAME, device=device)
    print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")

    # Extract texts
    texts = sample["alert_text"].tolist()
    print(f"  Texts to embed: {len(texts):,}")
    print(f"  Batch size: {BATCH_SIZE}")

    # Embed in batches with progress bar
    print(f"\nRunning embedding...")
    start_time = time.time()

    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,   
        device=device,
    )

    elapsed = time.time() - start_time
    print(f"\n  Embedding complete.")
    print(f"  Time elapsed       : {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
    print(f"  Embedding shape    : {embeddings.shape}")
    print(f"  Embedding dtype    : {embeddings.dtype}")
    print(f"  Sample vector norm : {np.linalg.norm(embeddings[0]):.4f} (should be ~1.0)")

    return embeddings

# Save outputs
def save_outputs(sample, embeddings):
    # Save embeddings as numpy array
    np.save(EMBEDDINGS, embeddings)
    print(f"\n  Embeddings saved to : {EMBEDDINGS}")
    print(f"  File size           : {EMBEDDINGS.stat().st_size / 1024**2:.1f} MB")

    # Save metadata for replication package
    meta = {
        "model":              MODEL_NAME,
        "embedding_dim":      int(embeddings.shape[1]),
        "n_alerts":           int(embeddings.shape[0]),
        "sample_per_tactic":  SAMPLE_PER_TACTIC,
        "benign_sample":      BENIGN_SAMPLE,
        "random_seed":        RANDOM_SEED,
        "batch_size":         BATCH_SIZE,
        "normalised":         True,
        "similarity_metric":  "cosine",
        "tactic_counts":      sample["attck_tactic"].value_counts().to_dict(),
        "technique_counts":   sample["attck_technique_id"].value_counts().to_dict(),
        "device":             "cuda" if torch.cuda.is_available() else "cpu",
        "gpu":                torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }
    with open(METADATA, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  Metadata saved to   : {METADATA}")

    # Quick sanity check
    print(f"\n  Sanity check — first 3 embeddings:")
    for i in range(min(3, len(sample))):
        row   = sample.iloc[i]
        vec   = embeddings[i]
        norm  = np.linalg.norm(vec)
        print(f"    [{i}] {row['attck_tactic']:25s} | "
              f"{row['attck_technique_id']:12s} | "
              f"norm={norm:.4f} | "
              f"text: {row['alert_text'][:60]}...")

# Main
def main():
    df         = load_data()
    sample     = stratified_sample(df)
    embeddings = embed_alerts(sample)
    save_outputs(sample, embeddings)

    print(f"\n{'='*65}")
    print(f"COMPLETE")
    print(f"Sample CSV  : {SAMPLE_CSV}")
    print(f"Embeddings  : {EMBEDDINGS}")
    print(f"Metadata    : {METADATA}")

if __name__ == "__main__":
    main()

Loading processed CIC-IDS data...
  Total alerts available: 435,290
  Tactics available:
attck_tactic
Discovery              158930
Benign                 130316
Impact                 128027
Credential Access       15342
Command And Control      2002
Execution                 652
Initial Access             21

Building stratified sample (seed=42)...

  Attack tactics to sample:
    Command And Control           :  2,002 available -> 2,000 sampled
    Credential Access             : 15,342 available -> 2,000 sampled
    Discovery                     : 158,930 available -> 2,000 sampled
    Execution                     :    652 available ->   652 sampled
    Impact                        : 128,027 available -> 2,000 sampled
    Initial Access                :     21 available ->    21 sampled
    Benign                        : 130,316 available -> 2,000 sampled

  Total sample size: 10,673
  Final tactic distribution:
attck_tactic
Discovery              2000
Credential Access      200

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dimension: 384
  Texts to embed: 10,673
  Batch size: 128

Running embedding...


Batches:   0%|          | 0/84 [00:00<?, ?it/s]


  Embedding complete.
  Time elapsed       : 10.2 seconds (0.2 minutes)
  Embedding shape    : (10673, 384)
  Embedding dtype    : float32
  Sample vector norm : 1.0000 (should be ~1.0)

  Embeddings saved to : ../embeddings/embeddings.npy
  File size           : 15.6 MB
  Metadata saved to   : ../embeddings/embedding_metadata.json

  Sanity check — first 3 embeddings:
    [0] Discovery                 | T1046        | norm=1.0000 | text: Network flow targeting port 1123 destination. Duration 38 mi...
    [1] Credential Access         | T1110.001    | norm=1.0000 | text: Network flow targeting FTP control destination. Duration 820...
    [2] Benign                    | BENIGN       | norm=1.0000 | text: Network flow targeting port 1580 destination. Duration 50 mi...

COMPLETE
Sample CSV  : ../embeddings/stratified_sample.csv
Embeddings  : ../embeddings/embeddings.npy
Metadata    : ../embeddings/embedding_metadata.json
